In [ ]:
# ===== Single-run GRU training (BEST + LAST ckpt) + (optional) AUGMENTATION + WARMUP FEATURE TRANSFORM =====
# KEY CHANGE: uses the COMPETITION METRIC (weighted Pearson) as the loss:
#   loss = -(pearson_y1 + pearson_y2)/2
#
# NEW: optional "feature-transform view" = per-sequence warmup normalisation using steps 0..98 (WARMUP_STEPS=99)
# This is deterministic (not augmentation) and can create a genuinely different ensemble member.

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------------------------
# Paths
# -------------------------
TRAIN_FILE = Path("/kaggle/input/lob-data/train.parquet")
VALID_FILE = Path("/kaggle/input/lob-data/valid.parquet")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# RUN CONFIG (edit these)
# -------------------------
SEED = 2024

# --- Preset A: GRU_h128_L4_d0.03 ---
HIDDEN = 128
NUM_LAYERS = 4
DROPOUT = 0.03

# --- Preset B: GRU_h64_L4_d0.05 ---
# HIDDEN = 64
# NUM_LAYERS = 4
# DROPOUT = 0.05

LR = 3e-4
WEIGHT_DECAY = 1e-5

# Feature-transform view (warmup normalisation)
USE_WARMUP_NORM = True
WARMUP_STEPS = 99          # steps 0..98
WARMUP_EPS = 1e-6

# -------------------------
# Model fixed
# -------------------------
INPUT_DIM = 32
D_OUT = 2

# -------------------------
# Training fixed
# -------------------------
BATCH_SIZE = 32
EPOCHS = 60
NUM_WORKERS = 2
PATIENCE = 6
CLIP_NORM = 1.0

# -------------------------
# Augmentation (TRAIN only)
# -------------------------
AUGMENT = False
DO_VAR_NORM = True
SCALE_LOW, SCALE_HIGH = 0.9, 1.1
NOISE_FRAC = 0.005
AUG_EPS = 1e-6

# -------------------------
# Repro
# -------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(SEED)

def _worker_init_fn(worker_id: int):
    s = SEED + worker_id * 1000
    np.random.seed(s)
    random.seed(s)

# -------------------------
# Dataset
# -------------------------
class SeqDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        augment: bool = False,
        global_feat_std: np.ndarray | None = None,
        use_warmup_norm: bool = False,
        warmup_steps: int = 99,
    ):
        self.augment = augment
        self.use_warmup_norm = bool(use_warmup_norm)
        self.warmup_steps = int(warmup_steps)

        self.groups = []
        for _, g in df.groupby("seq_ix", sort=False):
            g = g.sort_values("step_in_seq")
            x = g.iloc[:, 3:35].to_numpy(dtype=np.float32)   # 32 features
            y = g.iloc[:, 35:37].to_numpy(dtype=np.float32)  # 2 targets
            n = g["need_prediction"].to_numpy(dtype=np.uint8)
            self.groups.append((x, y, n))

        # compute global feature std (median over sequences), unless provided
        if global_feat_std is None:
            stds = []
            for x, _, _ in self.groups:
                stds.append(x.std(axis=0, ddof=0))
            stds = np.stack(stds, axis=0)
            self.global_feat_std = np.median(stds, axis=0).astype(np.float32)
        else:
            self.global_feat_std = global_feat_std.astype(np.float32)

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx: int):
        x, y, n = self.groups[idx]

        # ---- Feature transform view: warmup normalisation (deterministic, applies train+valid) ----
        if self.use_warmup_norm:
            x = x.copy()
            w = min(self.warmup_steps, x.shape[0])
            mu = x[:w].mean(axis=0, keepdims=True)
            sd = x[:w].std(axis=0, keepdims=True) + WARMUP_EPS
            x = (x - mu) / sd

        # ---- Augmentation (train only, stochastic) ----
        if self.augment and AUGMENT:
            x = x.copy()

            # (1) variance normalisation (per-feature)
            if DO_VAR_NORM:
                seq_std = x.std(axis=0, ddof=0)
                scale_to_target = self.global_feat_std / (seq_std + AUG_EPS)
                x *= scale_to_target[None, :]

            # (2) random global scaling
            s = np.random.uniform(SCALE_LOW, SCALE_HIGH)
            x *= np.float32(s)

            # (3) gaussian noise proportional to feature std
            noise_std = (NOISE_FRAC * self.global_feat_std)[None, :]
            x += (np.random.normal(0.0, 1.0, size=x.shape).astype(np.float32) * noise_std)

        return torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(n)

def collate_stack(batch):
    xs, ys, ns = zip(*batch)
    return torch.stack(xs, 0), torch.stack(ys, 0), torch.stack(ns, 0)

# -------------------------
# Model
# -------------------------
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden, num_layers, dropout, d_out):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            dropout=dropout if num_layers >= 2 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden, d_out)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.head(out)

# -------------------------
# Metric-as-loss (weighted Pearson)
# -------------------------
def weighted_pearson_1d(x: torch.Tensor, y: torch.Tensor, w: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    wsum = w.sum() + eps
    mx = (w * x).sum() / wsum
    my = (w * y).sum() / wsum
    xc = x - mx
    yc = y - my
    cov = (w * xc * yc).sum() / wsum
    vx = (w * xc * xc).sum() / wsum
    vy = (w * yc * yc).sum() / wsum
    return cov / (torch.sqrt(vx * vy) + eps)

def metric_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    w0 = target[:, 0].abs().clamp_min(1e-3)
    w1 = target[:, 1].abs().clamp_min(1e-3)
    c0 = weighted_pearson_1d(pred[:, 0], target[:, 0], w0, eps=eps)
    c1 = weighted_pearson_1d(pred[:, 1], target[:, 1], w1, eps=eps)
    return -0.5 * (c0 + c1)

# -------------------------
# Load data once
# -------------------------
print("Loading train/valid...")
train_df = pd.read_parquet(TRAIN_FILE)
valid_df = pd.read_parquet(VALID_FILE)

train_ds = SeqDataset(
    train_df,
    augment=True,
    global_feat_std=None,
    use_warmup_norm=USE_WARMUP_NORM,
    warmup_steps=WARMUP_STEPS,
)
valid_ds = SeqDataset(
    valid_df,
    augment=False,
    global_feat_std=train_ds.global_feat_std,
    use_warmup_norm=USE_WARMUP_NORM,
    warmup_steps=WARMUP_STEPS,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    collate_fn=collate_stack,
    worker_init_fn=_worker_init_fn if NUM_WORKERS > 0 else None,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    collate_fn=collate_stack,
)

print("DEVICE:", DEVICE, "| gpu:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu")
print(f"CONFIG: hidden={HIDDEN} layers={NUM_LAYERS} dropout={DROPOUT} lr={LR} wd={WEIGHT_DECAY} seed={SEED}")
print(f"WARMUP_NORM: {USE_WARMUP_NORM} | warmup_steps={WARMUP_STEPS}")
print(f"AUG: {AUGMENT} | var_norm={DO_VAR_NORM} | scale=[{SCALE_LOW},{SCALE_HIGH}] | noise_frac={NOISE_FRAC}")

# -------------------------
# Train one run
# -------------------------
tag = (
    f"h{HIDDEN}_L{NUM_LAYERS}_do{DROPOUT}_lr{LR:g}_wd{WEIGHT_DECAY:g}_bs{BATCH_SIZE}_seed{SEED}"
    f"_metricloss1_aug{int(AUGMENT)}_wn{int(USE_WARMUP_NORM)}"
)
best_path = OUT_DIR / f"gru_best_{tag}.pt"
last_path = OUT_DIR / f"gru_last_{tag}.pt"

model = GRUModel(INPUT_DIM, HIDDEN, NUM_LAYERS, DROPOUT, D_OUT).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_valid_loss = float("inf")
best_valid_metric = -float("inf")
bad = 0
last_valid_loss = None
last_valid_metric = None

for epoch in range(EPOCHS):
    model.train()
    train_sum = 0.0
    train_batches = 0

    for X, Y, N in train_loader:
        X = X.to(DEVICE, non_blocking=True)
        Y = Y.to(DEVICE, non_blocking=True)
        N = N.to(DEVICE, non_blocking=True)

        pred = model(X)
        mask = (N == 1)
        if mask.sum().item() == 0:
            continue

        p = pred[mask]
        t = Y[mask]

        loss = metric_loss(p, t)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        if CLIP_NORM and CLIP_NORM > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        opt.step()

        train_sum += loss.item()
        train_batches += 1

    train_avg = train_sum / max(1, train_batches)

    model.eval()
    with torch.no_grad():
        valid_sum = 0.0
        metric_sum = 0.0
        valid_batches = 0

        for X, Y, N in valid_loader:
            X = X.to(DEVICE, non_blocking=True)
            Y = Y.to(DEVICE, non_blocking=True)
            N = N.to(DEVICE, non_blocking=True)

            pred = model(X)
            mask = (N == 1)
            if mask.sum().item() == 0:
                continue

            p = pred[mask]
            t = Y[mask]

            vloss = metric_loss(p, t)
            valid_sum += vloss.item()
            metric_sum += (-vloss).item()
            valid_batches += 1

    valid_avg_loss = valid_sum / max(1, valid_batches)
    valid_avg_metric = metric_sum / max(1, valid_batches)

    last_valid_loss = valid_avg_loss
    last_valid_metric = valid_avg_metric

    if valid_avg_loss < best_valid_loss:
        best_valid_loss = valid_avg_loss
        best_valid_metric = valid_avg_metric
        bad = 0

        torch.save(
            {
                "state_dict": model.state_dict(),
                "input_dim": INPUT_DIM,
                "hidden": HIDDEN,
                "num_layers": NUM_LAYERS,
                "dropout": float(DROPOUT),
                "d_out": D_OUT,
                "epoch": epoch + 1,
                "best_valid_loss": float(best_valid_loss),
                "best_valid_metric": float(best_valid_metric),
                "tag": tag,
                "seed": int(SEED),
                "lr": float(LR),
                "weight_decay": float(WEIGHT_DECAY),
                "augment": bool(AUGMENT),
                "use_warmup_norm": bool(USE_WARMUP_NORM),
                "warmup_steps": int(WARMUP_STEPS),
                "loss_name": "metric_loss_weighted_pearson",
            },
            best_path,
        )
    else:
        bad += 1

    print(
        f"[{tag}] ep {epoch+1:02d}: "
        f"train_loss={train_avg:.4f} "
        f"valid_loss={valid_avg_loss:.4f} "
        f"valid_metric~={valid_avg_metric:.4f} "
        f"best_loss={best_valid_loss:.4f} bad={bad}/{PATIENCE}"
    )

    if bad >= PATIENCE:
        break

torch.save(
    {
        "state_dict": model.state_dict(),
        "input_dim": INPUT_DIM,
        "hidden": HIDDEN,
        "num_layers": NUM_LAYERS,
        "dropout": float(DROPOUT),
        "d_out": D_OUT,
        "epoch": epoch + 1,
        "valid_loss_last": float(last_valid_loss) if last_valid_loss is not None else None,
        "valid_metric_last": float(last_valid_metric) if last_valid_metric is not None else None,
        "tag": tag,
        "seed": int(SEED),
        "lr": float(LR),
        "weight_decay": float(WEIGHT_DECAY),
        "augment": bool(AUGMENT),
        "use_warmup_norm": bool(USE_WARMUP_NORM),
        "warmup_steps": int(WARMUP_STEPS),
        "loss_name": "metric_loss_weighted_pearson",
    },
    last_path,
)

print("\nSaved:")
print(" BEST:", best_path)
print(" LAST:", last_path)
print("Best valid loss (negative corr):", best_valid_loss)
print("Best valid metric approx (corr):", best_valid_metric)

In [ ]:
# ===== Window-based BiLSTM training (BEST + LAST ckpt) =====
# - Stateless: model sees a rolling window of K steps -> predicts last step only
# - Loss: competition metric (weighted Pearson) on the predicted last step
# - Uses need_prediction mask to ignore warmup / non-scored steps
#
# Paste into Kaggle and run. Edit RUN CONFIG only.

import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# -------------------------
# Paths
# -------------------------
TRAIN_FILE = Path("/kaggle/input/lob-data/train.parquet")
VALID_FILE = Path("/kaggle/input/lob-data/valid.parquet")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# RUN CONFIG (edit these)
# -------------------------
SEED = 42

WINDOW = 64              # try 32 or 64 first
USE_DELTAS = True        # concat (x_t - x_{t-1}) to inputs
HIDDEN = 64              # BiLSTM hidden
NUM_LAYERS = 2
DROPOUT = 0.10           # dropout inside LSTM (only active if num_layers>=2)

LR = 3e-4
WEIGHT_DECAY = 1e-5

BATCH_SIZE = 512         # windows are cheap; go larger if GPU allows
EPOCHS = 30
PATIENCE = 6
CLIP_NORM = 1.0
NUM_WORKERS = 2

# -------------------------
# Optional transforms (keep OFF initially)
# -------------------------
# (A) mild per-window standardisation (safe-ish); start False
WINDOW_ZSCORE = False
ZS_EPS = 1e-6

# (B) augmentation (start False; add later if stable)
AUGMENT = False
SCALE_LOW, SCALE_HIGH = 0.9, 1.1
NOISE_FRAC = 0.005

# -------------------------
# Repro
# -------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(SEED)

def _worker_init_fn(worker_id: int):
    s = SEED + worker_id * 1000
    np.random.seed(s)
    random.seed(s)

# -------------------------
# Metric-as-loss (weighted Pearson), torch version
# -------------------------
def weighted_pearson_1d(x: torch.Tensor, y: torch.Tensor, w: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    wsum = w.sum() + eps
    mx = (w * x).sum() / wsum
    my = (w * y).sum() / wsum
    xc = x - mx
    yc = y - my
    cov = (w * xc * yc).sum() / wsum
    vx = (w * xc * xc).sum() / wsum
    vy = (w * yc * yc).sum() / wsum
    return cov / (torch.sqrt(vx * vy) + eps)

def metric_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """
    pred, target: (N, 2)
    loss = -(corr_dim0 + corr_dim1)/2
    weights per dim = abs(target) clamped
    """
    w0 = target[:, 0].abs().clamp_min(1e-3)
    w1 = target[:, 1].abs().clamp_min(1e-3)
    c0 = weighted_pearson_1d(pred[:, 0], target[:, 0], w0, eps=eps)
    c1 = weighted_pearson_1d(pred[:, 1], target[:, 1], w1, eps=eps)
    return -0.5 * (c0 + c1)

# -------------------------
# Dataset: build windows from sequences
# -------------------------
class WindowDataset(Dataset):
    """
    Each item is a window ending at time t (inclusive):
      X_win: [WINDOW, D_in]  (raw + optional deltas)
      y_t:   [2]             (target at t)
      n_t:   0/1             (need_prediction at t)
    We will train only on windows with n_t == 1.
    """
    def __init__(self, df: pd.DataFrame, window: int, use_deltas: bool, augment: bool):
        self.window = window
        self.use_deltas = use_deltas
        self.augment = augment

        self.X = []      # list of np arrays (WINDOW, D_in)
        self.Y = []      # list of np arrays (2,)
        self.N = []      # list of uint8 (0/1)

        # Precompute global feature std for noise scaling (median over sequences)
        seq_stds = []

        for _, g in df.groupby("seq_ix", sort=False):
            g = g.sort_values("step_in_seq")

            x = g.iloc[:, 3:35].to_numpy(dtype=np.float32)    # [T, 32]
            y = g.iloc[:, 35:37].to_numpy(dtype=np.float32)   # [T, 2]
            n = g["need_prediction"].to_numpy(dtype=np.uint8) # [T]

            seq_stds.append(x.std(axis=0, ddof=0))

            T = x.shape[0]
            if T < window:
                continue

            # deltas (pad first delta with zeros so shapes match)
            if use_deltas:
                dx = np.zeros_like(x, dtype=np.float32)
                dx[1:] = x[1:] - x[:-1]
                feats = np.concatenate([x, dx], axis=1)  # [T, 64]
            else:
                feats = x  # [T, 32]

            # build rolling windows
            for t in range(window - 1, T):
                x_win = feats[t - window + 1 : t + 1]  # [WINDOW, D_in]
                self.X.append(x_win)
                self.Y.append(y[t])
                self.N.append(n[t])

        seq_stds = np.stack(seq_stds, axis=0)
        self.global_feat_std = np.median(seq_stds, axis=0).astype(np.float32)  # [32]
        # if deltas, just reuse same std for dx part (good enough)
        if self.use_deltas:
            self.global_feat_std = np.concatenate([self.global_feat_std, self.global_feat_std], axis=0).astype(np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx: int):
        x = self.X[idx].copy()
        y = self.Y[idx]
        n = self.N[idx]

        # Optional window z-score (start OFF)
        if WINDOW_ZSCORE:
            mu = x.mean(axis=0, keepdims=True)
            sd = x.std(axis=0, keepdims=True) + ZS_EPS
            x = (x - mu) / sd

        # Optional augmentation (start OFF)
        if self.augment and AUGMENT:
            s = np.random.uniform(SCALE_LOW, SCALE_HIGH)
            x *= np.float32(s)

            noise_std = (NOISE_FRAC * self.global_feat_std)[None, :]
            x += np.random.normal(0.0, 1.0, size=x.shape).astype(np.float32) * noise_std

        return torch.from_numpy(x), torch.from_numpy(y), torch.tensor(n, dtype=torch.uint8)

def collate_windows(batch):
    X, Y, N = zip(*batch)
    return torch.stack(X, 0), torch.stack(Y, 0), torch.stack(N, 0)

# -------------------------
# Model: BiLSTM -> last timestep -> linear head
# -------------------------
class BiLSTMWindowModel(nn.Module):
    def __init__(self, d_in: int, hidden: int, num_layers: int, dropout: float, d_out: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=d_in,
            hidden_size=hidden,
            num_layers=num_layers,
            dropout=dropout if num_layers >= 2 else 0.0,
            batch_first=True,
            bidirectional=True,
        )
        self.head = nn.Linear(2 * hidden, d_out)  # bidirectional -> 2*hidden

    def forward(self, x):
        # x: [B, W, d_in]
        out, _ = self.lstm(x)       # out: [B, W, 2H]
        last = out[:, -1, :]        # [B, 2H]
        y = self.head(last)         # [B, 2]
        return y

# -------------------------
# Load data + build datasets
# -------------------------
print("Loading train/valid...")
train_df = pd.read_parquet(TRAIN_FILE)
valid_df = pd.read_parquet(VALID_FILE)

print("Building window datasets...")
train_ds = WindowDataset(train_df, window=WINDOW, use_deltas=USE_DELTAS, augment=True)
valid_ds = WindowDataset(valid_df, window=WINDOW, use_deltas=USE_DELTAS, augment=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    collate_fn=collate_windows,
    worker_init_fn=_worker_init_fn if NUM_WORKERS > 0 else None,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE == "cuda"),
    collate_fn=collate_windows,
)

d_in = 32 * (2 if USE_DELTAS else 1)
print("DEVICE:", DEVICE, "| gpu:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu")
print(f"CONFIG: window={WINDOW} d_in={d_in} deltas={USE_DELTAS} hidden={HIDDEN} layers={NUM_LAYERS} dropout={DROPOUT}")
print(f"LOSS: metric_loss (weighted pearson)")
print(f"EXTRAS: zscore={WINDOW_ZSCORE} aug={AUGMENT} scale=[{SCALE_LOW},{SCALE_HIGH}] noise_frac={NOISE_FRAC}")

# -------------------------
# Train
# -------------------------
tag = (
    f"bilstmW{WINDOW}_din{d_in}_dx{int(USE_DELTAS)}"
    f"_h{HIDDEN}_L{NUM_LAYERS}_do{DROPOUT}"
    f"_lr{LR:g}_wd{WEIGHT_DECAY:g}"
    f"_bs{BATCH_SIZE}_seed{SEED}"
    f"_z{int(WINDOW_ZSCORE)}_aug{int(AUGMENT)}"
)

best_path = OUT_DIR / f"bilstm_best_{tag}.pt"
last_path = OUT_DIR / f"bilstm_last_{tag}.pt"

model = BiLSTMWindowModel(d_in=d_in, hidden=HIDDEN, num_layers=NUM_LAYERS, dropout=DROPOUT, d_out=2).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_valid_loss = float("inf")
best_valid_metric = -float("inf")
bad = 0
last_valid_loss = None
last_valid_metric = None

for epoch in range(EPOCHS):
    # ---- train ----
    model.train()
    train_sum = 0.0
    train_batches = 0

    for X, Y, N in train_loader:
        X = X.to(DEVICE, non_blocking=True)
        Y = Y.to(DEVICE, non_blocking=True)
        N = N.to(DEVICE, non_blocking=True)

        # only train on need_prediction==1
        mask = (N == 1)
        if mask.sum().item() == 0:
            continue

        pred = model(X)        # [B,2]
        p = pred[mask]
        t = Y[mask]

        loss = metric_loss(p, t)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        if CLIP_NORM and CLIP_NORM > 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
        opt.step()

        train_sum += loss.item()
        train_batches += 1

    train_avg = train_sum / max(1, train_batches)

    # ---- valid ----
    model.eval()
    with torch.no_grad():
        valid_sum = 0.0
        metric_sum = 0.0
        valid_batches = 0

        for X, Y, N in valid_loader:
            X = X.to(DEVICE, non_blocking=True)
            Y = Y.to(DEVICE, non_blocking=True)
            N = N.to(DEVICE, non_blocking=True)

            mask = (N == 1)
            if mask.sum().item() == 0:
                continue

            pred = model(X)
            p = pred[mask]
            t = Y[mask]

            vloss = metric_loss(p, t)      # negative corr
            valid_sum += vloss.item()
            metric_sum += (-vloss).item()  # approx corr
            valid_batches += 1

    valid_avg_loss = valid_sum / max(1, valid_batches)
    valid_avg_metric = metric_sum / max(1, valid_batches)

    last_valid_loss = valid_avg_loss
    last_valid_metric = valid_avg_metric

    # checkpoint on best (lowest) valid loss
    if valid_avg_loss < best_valid_loss:
        best_valid_loss = valid_avg_loss
        best_valid_metric = valid_avg_metric
        bad = 0

        torch.save(
            {
                "state_dict": model.state_dict(),
                "window": int(WINDOW),
                "use_deltas": bool(USE_DELTAS),
                "d_in": int(d_in),
                "hidden": int(HIDDEN),
                "num_layers": int(NUM_LAYERS),
                "dropout": float(DROPOUT),
                "d_out": 2,
                "epoch": epoch + 1,
                "best_valid_loss": float(best_valid_loss),
                "best_valid_metric": float(best_valid_metric),
                "tag": tag,
                "seed": int(SEED),
                "lr": float(LR),
                "weight_decay": float(WEIGHT_DECAY),
                "window_zscore": bool(WINDOW_ZSCORE),
                "augment": bool(AUGMENT),
                "scale_low": float(SCALE_LOW),
                "scale_high": float(SCALE_HIGH),
                "noise_frac": float(NOISE_FRAC),
                "loss_name": "metric_loss_weighted_pearson",
                "model_type": "bilstm_window",
            },
            best_path,
        )
    else:
        bad += 1

    print(
        f"[{tag}] ep {epoch+1:02d}: "
        f"train_loss={train_avg:.4f} "
        f"valid_loss={valid_avg_loss:.4f} "
        f"valid_metric~={valid_avg_metric:.4f} "
        f"best_loss={best_valid_loss:.4f} bad={bad}/{PATIENCE}"
    )

    if bad >= PATIENCE:
        break

torch.save(
    {
        "state_dict": model.state_dict(),
        "window": int(WINDOW),
        "use_deltas": bool(USE_DELTAS),
        "d_in": int(d_in),
        "hidden": int(HIDDEN),
        "num_layers": int(NUM_LAYERS),
        "dropout": float(DROPOUT),
        "d_out": 2,
        "epoch": epoch + 1,
        "valid_loss_last": float(last_valid_loss) if last_valid_loss is not None else None,
        "valid_metric_last": float(last_valid_metric) if last_valid_metric is not None else None,
        "tag": tag,
        "seed": int(SEED),
        "lr": float(LR),
        "weight_decay": float(WEIGHT_DECAY),
        "window_zscore": bool(WINDOW_ZSCORE),
        "augment": bool(AUGMENT),
        "scale_low": float(SCALE_LOW),
        "scale_high": float(SCALE_HIGH),
        "noise_frac": float(NOISE_FRAC),
        "loss_name": "metric_loss_weighted_pearson",
        "model_type": "bilstm_window",
    },
    last_path,
)

print("\nSaved:")
print(" BEST:", best_path)
print(" LAST:", last_path)
print("Best valid loss (negative corr):", best_valid_loss)
print("Best valid metric approx (corr):", best_valid_metric)

In [ ]:
# ===== Stateful LSTM with Data Augmentation + Highway Head =====
# Based on 5th place solution
# - Stateful LSTM (hidden state carries across sequences)
# - 5x data augmentation (their biggest win: +0.0035 R²)
# - Highway Head output (their +0.00116 R²)
# - Feature view: sigmoid + silu (for ensemble diversity vs BiLSTM's tanh+sigmoid)

import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# -------------------------
# Paths
# -------------------------
TRAIN_FILE = Path("/kaggle/input/lob-data/train.parquet")
VALID_FILE = Path("/kaggle/input/lob-data/valid.parquet")
OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# RUN CONFIG
# -------------------------
SEED = 42

USE_DELTAS = True  # [x, dx] → d_in = 64
HIDDEN = 150       # 5th place used 150 for LSTM
NUM_LAYERS = 1     # Single layer stateful

# Feature view for diversity (different from BiLSTM's tanh_sigmoid)
FEATURE_VIEW = "sigmoid_silu"  # options: "none", "sigmoid_silu"

# AUGMENTATION - THEIR BIGGEST WIN
AUGMENT = True
AUG_MULTIPLIER = 5  # 5x dataset size
SCALE_LOW, SCALE_HIGH = 0.6, 1.4  # Their exact params
NOISE_FRAC = 0.05  # 5% noise (not 0.005!)

# Training
LR = 3e-4
WEIGHT_DECAY = 0.1  # They used 0.1 weight decay
BATCH_SIZE = 32     # Stateful needs small batch
EPOCHS = 64         # They used multi-stage: 64 epochs first stage
PATIENCE = 8
CLIP_NORM = 1.0
DROPOUT = 0.1       # For highway head

NUM_WORKERS = 2
PIN_MEMORY = (DEVICE == "cuda")

# -------------------------
# Repro
# -------------------------
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(SEED)

def _worker_init_fn(worker_id: int):
    s = SEED + worker_id * 1000
    np.random.seed(s)
    random.seed(s)

# -------------------------
# Metric: weighted Pearson
# -------------------------
def weighted_pearson_1d_np(x: np.ndarray, y: np.ndarray, w: np.ndarray, eps: float = 1e-12) -> float:
    w = w.astype(np.float64)
    x = x.astype(np.float64)
    y = y.astype(np.float64)
    wsum = w.sum() + eps
    mx = (w * x).sum() / wsum
    my = (w * y).sum() / wsum
    xc = x - mx
    yc = y - my
    cov = (w * xc * yc).sum() / wsum
    vx = (w * xc * xc).sum() / wsum
    vy = (w * yc * yc).sum() / wsum
    return float(cov / (np.sqrt(vx * vy) + eps))

# -------------------------
# Stateful Dataset with Augmentation
# -------------------------
class StatefulSequenceDataset(Dataset):
    """
    Each sequence is an item. Returns full sequence.
    Augmentation: 5x dataset by variance norm + scale + noise
    """
    def __init__(self, df: pd.DataFrame, use_deltas: bool, augment: bool, aug_multiplier: int = 5):
        self.use_deltas = use_deltas
        self.augment = augment
        self.aug_multiplier = aug_multiplier if augment else 1
        
        self.sequences = []
        seq_stds = []
        
        for _, g in df.groupby("seq_ix", sort=False):
            g = g.sort_values("step_in_seq")
            
            x = g.iloc[:, 3:35].to_numpy(np.float32)  # [T, 32]
            y = g.iloc[:, 35:37].to_numpy(np.float32)  # [T, 2]
            n = g["need_prediction"].to_numpy(np.uint8)  # [T]
            
            # Compute std for augmentation
            if augment:
                seq_stds.append(x.std(axis=0, ddof=0))
            
            # Add deltas
            if use_deltas:
                dx = np.zeros_like(x, dtype=np.float32)
                dx[1:] = x[1:] - x[:-1]
                x = np.concatenate([x, dx], axis=1)  # [T, 64]
            
            self.sequences.append((x, y, n))
        
        # Global median std for augmentation
        if augment and len(seq_stds) > 0:
            seq_stds = np.stack(seq_stds, axis=0)  # [N_seq, 32]
            self.global_std = np.median(seq_stds, axis=0).astype(np.float32)  # [32]
            
            if use_deltas:
                # Duplicate for deltas
                self.global_std = np.concatenate([self.global_std, self.global_std], axis=0)
        else:
            self.global_std = None
    
    def __len__(self):
        return len(self.sequences) * self.aug_multiplier
    
    def __getitem__(self, idx: int):
        seq_idx = idx // self.aug_multiplier
        aug_idx = idx % self.aug_multiplier
        
        x, y, n = self.sequences[seq_idx]
        x = x.copy()  # Don't modify original
        
        # Apply augmentation (skip first copy = original data)
        if self.augment and aug_idx > 0 and self.global_std is not None:
            # 1. Normalize to median variance
            seq_std = x.std(axis=0, ddof=0) + 1e-8
            x = x * (self.global_std / seq_std)[None, :]
            
            # 2. Random scaling U(0.6, 1.4)
            scale = np.random.uniform(SCALE_LOW, SCALE_HIGH)
            x = x * scale
            
            # 3. Gaussian noise injection (5% of global std)
            noise = np.random.normal(0.0, 1.0, size=x.shape).astype(np.float32)
            noise = noise * (NOISE_FRAC * self.global_std)[None, :]
            x = x + noise
        
        return (
            torch.from_numpy(x),
            torch.from_numpy(y),
            torch.from_numpy(n)
        )

def collate_sequences(batch):
    """Collate variable-length sequences with padding"""
    Xs, Ys, Ns = zip(*batch)
    
    # Find max length
    max_len = max(x.size(0) for x in Xs)
    batch_size = len(Xs)
    d_in = Xs[0].size(1)
    
    # Pad sequences
    X_pad = torch.zeros(batch_size, max_len, d_in, dtype=torch.float32)
    Y_pad = torch.zeros(batch_size, max_len, 2, dtype=torch.float32)
    N_pad = torch.zeros(batch_size, max_len, dtype=torch.uint8)
    lengths = torch.zeros(batch_size, dtype=torch.long)
    
    for i, (x, y, n) in enumerate(zip(Xs, Ys, Ns)):
        seq_len = x.size(0)
        X_pad[i, :seq_len] = x
        Y_pad[i, :seq_len] = y
        N_pad[i, :seq_len] = n
        lengths[i] = seq_len
    
    return X_pad, Y_pad, N_pad, lengths

# -------------------------
# Highway Head (from 5th place report)
# -------------------------
class HighwayHead(nn.Module):
    """
    Adaptive gating between non-linear and linear paths.
    From equation in technical report.
    """
    def __init__(self, d_in: int, d_mid: int, d_out: int, dropout: float = 0.1):
        super().__init__()
        self.ln = nn.LayerNorm(d_in)
        self.W_h = nn.Linear(d_in, d_mid)  # Non-linear transform
        self.W_t = nn.Linear(d_in, d_mid)  # Transform gate
        self.W_x = nn.Linear(d_in, d_mid)  # Linear projection
        self.W_out = nn.Linear(d_mid, d_out)
        self.dropout = nn.Dropout(dropout)
        
        # Initialize transform gate to -1 (bias towards passthrough initially)
        nn.init.constant_(self.W_t.bias, -1.0)
    
    def forward(self, x):
        # x: [..., d_in]
        x_norm = self.ln(x)
        
        # Non-linear path
        h = F.gelu(self.W_h(x_norm))
        
        # Transform gate
        t = torch.sigmoid(self.W_t(x_norm))
        
        # Linear projection
        x_s = self.W_x(x_norm)
        
        # Highway aggregation: t * h + (1-t) * x_s
        y_mid = t * h + (1 - t) * x_s
        
        # Final projection with dropout
        y_out = self.W_out(self.dropout(y_mid))
        
        return y_out

# -------------------------
# Stateful LSTM Model
# -------------------------
class StatefulLSTMModel(nn.Module):
    def __init__(self, d_in: int, hidden: int, d_out: int = 2, 
                 feature_view: str = "none", dropout: float = 0.1):
        super().__init__()
        self.hidden = hidden
        self.feature_view = feature_view
        
        # Feature transform doubles input dimension
        if feature_view == "sigmoid_silu":
            lstm_in = 2 * d_in
        else:
            lstm_in = d_in
        
        self.lstm = nn.LSTM(
            input_size=lstm_in,
            hidden_size=hidden,
            num_layers=1,
            batch_first=True,
        )
        
        # Highway head instead of simple Linear
        self.head = HighwayHead(hidden, hidden, d_out, dropout=dropout)
    
    def forward(self, x, hidden=None):
        # x: [B, T, d_in]
        
        # Apply feature view transform
        if self.feature_view == "sigmoid_silu":
            x = torch.cat([torch.sigmoid(x), F.silu(x)], dim=-1)
        
        # LSTM forward
        out, hidden = self.lstm(x, hidden)
        
        # Highway head
        pred = self.head(out)  # [B, T, 2]
        
        return pred, hidden

# -------------------------
# Chrono Initialization (from report)
# -------------------------
def chrono_init_lstm(model, T_max=1000):
    """Initialize LSTM forget gate biases with log-uniform distribution"""
    for name, param in model.named_parameters():
        if 'lstm' in name and 'bias' in name:
            # LSTM bias order: [input, forget, cell, output]
            # We want to initialize forget gate bias (second quarter)
            n = param.size(0)
            forget_size = n // 4
            start_idx = forget_size  # Second quarter
            
            # Sample from log-uniform: log(U(1, T_max) - 1)
            forget_bias = torch.log(torch.FloatTensor(forget_size).uniform_(1, T_max) - 1)
            param.data[start_idx:start_idx + forget_size] = forget_bias

# -------------------------
# Training with stateful hidden states
# -------------------------
@torch.no_grad()
def evaluate_stateful(model, loader, device):
    """Evaluate on validation set with stateful processing"""
    model.eval()
    
    all_preds = []
    all_targets = []
    all_weights = []
    
    for X, Y, N, lengths in loader:
        X = X.to(device)
        Y = Y.to(device)
        N = N.to(device)
        
        hidden = None
        pred, hidden = model(X, hidden)
        
        # Extract predictions where need_prediction == 1
        for b in range(X.size(0)):
            seq_len = lengths[b].item()
            mask = N[b, :seq_len] == 1
            
            if mask.sum() > 0:
                p = pred[b, :seq_len][mask].cpu().numpy()
                t = Y[b, :seq_len][mask].cpu().numpy()
                w = np.maximum(np.abs(t), 1e-3)
                
                all_preds.append(p)
                all_targets.append(t)
                all_weights.append(w)
    
    # Concatenate all
    all_preds = np.concatenate(all_preds, axis=0)  # [N, 2]
    all_targets = np.concatenate(all_targets, axis=0)  # [N, 2]
    all_weights = np.concatenate(all_weights, axis=0)  # [N, 2]
    
    # Compute weighted Pearson per output
    c0 = weighted_pearson_1d_np(all_preds[:, 0], all_targets[:, 0], all_weights[:, 0])
    c1 = weighted_pearson_1d_np(all_preds[:, 1], all_targets[:, 1], all_weights[:, 1])
    
    return 0.5 * (c0 + c1)

# -------------------------
# Load data
# -------------------------
print("Loading train/valid...")
train_df = pd.read_parquet(TRAIN_FILE)
valid_df = pd.read_parquet(VALID_FILE)

print("Building datasets...")
train_ds = StatefulSequenceDataset(
    train_df, 
    use_deltas=USE_DELTAS, 
    augment=AUGMENT, 
    aug_multiplier=AUG_MULTIPLIER
)
valid_ds = StatefulSequenceDataset(
    valid_df, 
    use_deltas=USE_DELTAS, 
    augment=False,  # No augmentation for validation
    aug_multiplier=1
)

d_in = 32 * (2 if USE_DELTAS else 1)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_sequences,
    worker_init_fn=_worker_init_fn if NUM_WORKERS > 0 else None,
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_sequences,
)

print(f"DEVICE: {DEVICE}")
print(f"CONFIG: d_in={d_in} hidden={HIDDEN} deltas={USE_DELTAS} view={FEATURE_VIEW}")
print(f"AUGMENT: {AUGMENT} (multiplier={AUG_MULTIPLIER}, scale=[{SCALE_LOW},{SCALE_HIGH}], noise={NOISE_FRAC})")
print(f"Dataset sizes: train={len(train_ds):,} (base={len(train_ds)//AUG_MULTIPLIER}), valid={len(valid_ds):,}")

# -------------------------
# Initialize model
# -------------------------
model = StatefulLSTMModel(
    d_in=d_in,
    hidden=HIDDEN,
    d_out=2,
    feature_view=FEATURE_VIEW,
    dropout=DROPOUT,
).to(DEVICE)

# Apply Chrono initialization
chrono_init_lstm(model, T_max=1000)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
mse = nn.MSELoss()

# -------------------------
# Training loop
# -------------------------
tag = (
    f"lstm_stateful_h{HIDDEN}_din{d_in}_dx{int(USE_DELTAS)}"
    f"_view{FEATURE_VIEW}_aug{int(AUGMENT)}x{AUG_MULTIPLIER}"
    f"_lr{LR:g}_wd{WEIGHT_DECAY:g}_bs{BATCH_SIZE}_seed{SEED}"
)

best_path = OUT_DIR / f"lstm_best_{tag}.pt"
last_path = OUT_DIR / f"lstm_last_{tag}.pt"

best_metric = -1e9
bad = 0

print(f"\nStarting training: {tag}\n")

for epoch in range(EPOCHS):
    model.train()
    train_loss_sum = 0.0
    train_batches = 0
    
    for X, Y, N, lengths in train_loader:
        X = X.to(DEVICE)
        Y = Y.to(DEVICE)
        N = N.to(DEVICE)
        
        # Forward pass (stateful within batch)
        hidden = None
        pred, hidden = model(X, hidden)
        
        # Compute loss only on need_prediction positions
        mask = N == 1
        if mask.sum() > 0:
            loss = mse(pred[mask], Y[mask])
            
            opt.zero_grad(set_to_none=True)
            loss.backward()
            
            if CLIP_NORM and CLIP_NORM > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            
            opt.step()
            
            train_loss_sum += loss.item()
            train_batches += 1
    
    train_mse = train_loss_sum / max(1, train_batches)
    
    # Evaluate on validation
    val_metric = evaluate_stateful(model, valid_loader, DEVICE)
    
    # Save best
    improved = val_metric > best_metric
    if improved:
        best_metric = val_metric
        bad = 0
        torch.save({
            "state_dict": model.state_dict(),
            "d_in": d_in,
            "hidden": HIDDEN,
            "use_deltas": USE_DELTAS,
            "feature_view": FEATURE_VIEW,
            "dropout": DROPOUT,
            "augment": AUGMENT,
            "aug_multiplier": AUG_MULTIPLIER,
            "epoch": epoch + 1,
            "best_valid_metric": float(best_metric),
            "tag": tag,
            "seed": SEED,
        }, best_path)
    else:
        bad += 1
    
    print(
        f"[{tag}] ep {epoch+1:02d}: train_mse={train_mse:.6f} "
        f"valid_metric={val_metric:.5f} best={best_metric:.5f} bad={bad}/{PATIENCE}"
    )
    
    if bad >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Save last checkpoint
torch.save({
    "state_dict": model.state_dict(),
    "d_in": d_in,
    "hidden": HIDDEN,
    "use_deltas": USE_DELTAS,
    "feature_view": FEATURE_VIEW,
    "dropout": DROPOUT,
    "augment": AUGMENT,
    "aug_multiplier": AUG_MULTIPLIER,
    "epoch": epoch + 1,
    "last_valid_metric": float(val_metric),
    "tag": tag,
    "seed": SEED,
}, last_path)

print("\n" + "="*80)
print("Training complete!")
print(f"Best validation metric: {best_metric:.5f}")
print(f"Saved to: {best_path}")
print("="*80)

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn

# Add the Kaggle "Model" input folder that contains utils.py
sys.path.append("/kaggle/input/gru2/other/default/1")

from utils import DataPoint, ScorerStepByStep

DEVICE = "cpu"  # submission is CPU, scoring on CPU is fine

class GRUModel(nn.Module):
    def __init__(self, input_dim=32, hidden=128, num_layers=4, dropout=0.1, d_out=2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            dropout=dropout if num_layers >= 2 else 0.0,
            batch_first=True,
        )
        self.head = nn.Linear(hidden, d_out)

    def forward(self, x, h=None):
        out, h = self.gru(x, h)
        y = self.head(out)
        return y, h

class PredictionModel:
    def __init__(self, ckpt_path: str):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)

        self.model = GRUModel(
            input_dim=int(ckpt.get("input_dim", 32)),
            hidden=int(ckpt.get("hidden", 128)),
            num_layers=int(ckpt.get("num_layers", 4)),
            dropout=float(ckpt.get("dropout", 0.1)),
            d_out=2,
        ).to(DEVICE)

        self.model.load_state_dict(ckpt["state_dict"])
        self.model.eval()

        self.current_seq_ix = None
        self.h = None

    def _reset(self, seq_ix: int):
        self.current_seq_ix = seq_ix
        self.h = None

    @torch.no_grad()
    def predict(self, data_point: DataPoint):
        if self.current_seq_ix != data_point.seq_ix:
            self._reset(data_point.seq_ix)

        x = torch.from_numpy(data_point.state.astype(np.float32, copy=False)).to(DEVICE).view(1, 1, -1)
        y, self.h = self.model(x, self.h)
        pred = y[0, 0].cpu().numpy().astype(np.float32)

        if not data_point.need_prediction:
            return None

        return np.clip(pred, -6.0, 6.0).astype(np.float32)

# pick BEST checkpoint automatically
best = sorted(Path("/kaggle/working").glob("gru_best_*.pt"))
last = sorted(Path("/kaggle/working").glob("gru_last_*.pt"))

ckpt_path = str(best[-1] if best else last[-1])
print("Using ckpt:", ckpt_path)

test_file = "/kaggle/input/lob-data/valid.parquet"

model = PredictionModel(ckpt_path)
scorer = ScorerStepByStep(test_file)

print("Scoring...")
results = scorer.score(model)
print("Results:", results)